# バッチ予測を伴う AutoML Agent Platform Pipelines による継続的トレーニング

**学習目標:**
1. Agent Platform AutoML の事前構築済みコンポーネントの使用方法を学ぶ
1. BigQuery をデータソースとして使用し、これらのコンポーネントで Agent Platform AutoML パイプラインを構築する方法を学ぶ
1. Agent Platform AutoML パイプラインをコンパイル、アップロード、実行する方法を学ぶ
1. AutoML パイプラインから BigQuery ソースを使用したバッチ予測を提供する


このラボでは、モデルを使用して BigQuery へのバッチ予測のトレーニング、調整、提供を行う **Agent Platform AutoML AI** サービスをオーケストレーションする Agent Platform AutoML パイプラインを構築、デプロイ、実行します。

## セットアップ

In [ ]:
import os

from google.cloud import aiplatform

%load_ext google.cloud.bigquery

In [2]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value project)
PROJECT = PROJECT[0]

os.environ["PROJECT"] = PROJECT

### BigQuery データ

KFP ウォークスルー ラボをまだ完了していない場合は、次のセルを実行して、このラボに必要なデータを含む BigQuery データセットとテーブルを作成する必要があります。

**注:** すでに `<PROJECT_ID>.covertype_dataset.covertype` の BigQuery テーブルに Covertype データが存在する場合は、**パイプライン設計の理解** にスキップできます。

In [ ]:
%%bash

DATASET_LOCATION=US
DATASET_ID=covertype_dataset
TABLE_ID=covertype
DATA_SOURCE=gs://asl-public-data/data/covertype/dataset.csv
SCHEMA=Elevation:INTEGER,\
Aspect:INTEGER,\
Slope:INTEGER,\
Horizontal_Distance_To_Hydrology:INTEGER,\
Vertical_Distance_To_Hydrology:INTEGER,\
Horizontal_Distance_To_Roadways:INTEGER,\
Hillshade_9am:INTEGER,\
Hillshade_Noon:INTEGER,\
Hillshade_3pm:INTEGER,\
Horizontal_Distance_To_Fire_Points:INTEGER,\
Wilderness_Area:STRING,\
Soil_Type:STRING,\
Cover_Type:INTEGER

bq --location=$DATASET_LOCATION --project_id=$PROJECT mk --dataset $DATASET_ID

bq --project_id=$PROJECT --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

## パイプライン設計の理解

パイプラインによって実装されるワークフローは、Python ベースのドメイン固有言語（DSL）を使用して定義されます。パイプラインの DSL は、以下で生成する `pipeline/pipeline_automl_batch_preds.py` ファイルに記述されています。

パイプラインの DSL は、ファイルパスや接続文字列などの環境固有の設定をハードコードしないように設計されています。これらの設定は、一連の環境変数を介してパイプライン コードに提供されます。

## パイプラインの構築とデプロイ

パイプラインをディスクに書き込みましょう。

In [ ]:
%%writefile ./pipeline/pipeline_automl_batch_preds.py
"""Kubeflow Covertype パイプライン。"""

import os

from google_cloud_pipeline_components.v1.automl.training_job import (
    AutoMLTabularTrainingJobRunOp
)

from google_cloud_pipeline_components.v1.dataset import (
    TabularDatasetCreateOp
)

from google_cloud_pipeline_components.v1.batch_predict_job import (
    ModelBatchPredictOp
)

from kfp import dsl

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT = os.getenv("PROJECT")
DATASET_SOURCE = os.getenv("DATASET_SOURCE")
PIPELINE_NAME = os.getenv("PIPELINE_NAME", "covertype")
DISPLAY_NAME = os.getenv("MODEL_DISPLAY_NAME", PIPELINE_NAME)
TARGET_COLUMN = os.getenv("TARGET_COLUMN", "Cover_Type")
BATCH_PREDS_SOURCE_URI = os.getenv("BATCH_PREDS_SOURCE_URI")
# Agent Platform AutoML コンポーネントのイメージ バージョン
KFP_IMAGE = "gcr.io/ml-pipeline/google-cloud-pipeline-components:2.16.1"

@dsl.pipeline(
    name=f"{PIPELINE_NAME}-automl-pipeline-batch-preds",
    description=f"AutoML Pipeline for {PIPELINE_NAME}",
    pipeline_root=PIPELINE_ROOT,
)
def create_pipeline():

    dataset_create_task = TabularDatasetCreateOp(
        display_name=DISPLAY_NAME,
        bq_source=DATASET_SOURCE,
        project=PROJECT,
    )

    automl_training_task = AutoMLTabularTrainingJobRunOp(
        project=PROJECT,
        display_name=DISPLAY_NAME,
        optimization_prediction_type="classification",
        dataset=dataset_create_task.outputs["dataset"],
        target_column=TARGET_COLUMN,
    )

    # コンテナ イメージのバージョンをオーバーライド
    automl_training_task.set_container_image(KFP_IMAGE)

    batch_predict_op = ModelBatchPredictOp(
        project=PROJECT,
        job_display_name="batch_predict_job",
        model=automl_training_task.outputs["model"],
        bigquery_source_input_uri=BATCH_PREDS_SOURCE_URI,
        instances_format="bigquery",
        predictions_format="bigquery",
        bigquery_destination_output_uri=f'bq://{PROJECT}',
    )


./pipeline/pipeline_automl_batch_preds.py を上書き中

### ModelBatchPredictOp の理解
AutoML Tabular モデルを扱う場合、`ModelBatchPredictOp` は次の入力を受け取ることができます。
* `model`: バッチ予測の提供に使用するモデル リソース
* `bigquery_source_uri`: `bq://PROJECT.DATASET.TABLE` 形式の、バッチ予測を提供するサンプルが含まれる BigQuery テーブルへの URI
* `instances_format`: BigQuery データに対してバッチ予測を提供するための "bigquery"
* `predictions_format`: バッチ予測の結果を BigQuery に保存するための "bigquery"
* `bigquery_destination_output_uri`: `bq://PROJECT_ID` 形式。これはバッチ予測の結果が保存されるプロジェクトです。`ModelBatchPredictOp` はこのプロジェクトにデータセットを作成します。

`ModelBatchPredictOp` の完了後、`prediction_<model-display-name>_<job-create-time>` という名前の新しい BigQuery データセットが作成されます。このデータセット内には、バッチ予測のサンプルと予測されたラベルを含む `predictions` テーブルがあります。バッチ予測でエラーが発生した場合、`errors` テーブルも作成されます。`errors` テーブルには予測に失敗した行が含まれます。

### バッチ予測用データを含む BigQuery テーブルの作成
パイプラインをコンパイルして実行する前に、バッチ予測を提供したいデータを含む BigQuery テーブルを作成しましょう。「新しい」データをシミュレートするために、ラベル以外のすべての列について既存のテーブルにクエリを実行し、`newdata` というテーブルを作成します。このテーブルへの URI が、`ModelBatchPredictOp` への `bigquery_source_input_uri` 入力となります。

In [6]:
%%bigquery
CREATE OR REPLACE TABLE covertype_dataset.newdata AS 
SELECT * EXCEPT(Cover_Type)
FROM covertype_dataset.covertype
LIMIT 10000

Query complete after 0.01s: 100%|██████████| 3/3 [00:00<00:00, 1720.86query/s]                        


""


### パイプラインのコンパイル

まず、パイプライン コンパイラに渡される環境変数を定義しましょう。

In [7]:
ARTIFACT_STORE = f"gs://{PROJECT}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATASET_SOURCE = f"bq://{PROJECT}.covertype_dataset.covertype"
BATCH_PREDS_SOURCE_URI = f"bq://{PROJECT}.covertype_dataset.newdata"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT={PROJECT}
%env REGION={REGION}
%env DATASET_SOURCE={DATASET_SOURCE}
%env BATCH_PREDS_SOURCE_URI={BATCH_PREDS_SOURCE_URI}

env: PIPELINE_ROOT=gs://kylesteckler-demo-kfp-artifact-store/pipeline
env: PROJECT=kylesteckler-demo
env: REGION=us-central1
env: DATASET_SOURCE=bq://kylesteckler-demo.covertype_dataset.covertype
env: BATCH_PREDS_SOURCE_URI=bq://kylesteckler-demo.covertype_dataset.newdata


`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう。

In [8]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

gs://kylesteckler-demo-kfp-artifact-store/


#### CLI コンパイラを使用したパイプラインのコンパイル

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします。

In [ ]:
PIPELINE_YAML = "covertype_automl_pipeline_batch_preds.yaml"

In [ ]:
!kfp dsl compile --py pipeline/pipeline_automl_batch_preds.py --output $PIPELINE_YAML

/home/jupyter/asl-ml-immersion/notebooks/kubeflow_pipelines/pipelines/solutions/covertype_automl_pipeline_batch_preds.yaml

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます。

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)
```

結果としてパイプライン ファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプライン パッケージのデプロイ

In [ ]:
aiplatform.init(project=PROJECT, location=REGION)

pipeline = aiplatform.PipelineJob(
    display_name="automl_covertype_kfp_pipeline_batch_predictions",
    template_path=PIPELINE_YAML,
    enable_caching=True,
)

pipeline.run()

### BatchPredictOp によって作成されたリソースの理解

パイプラインの実行が完了すると、`prediction_<model-display-name>_<job-create-time>` という名前の新しい BigQuery データセットが表示されます。このデータセット内には、バッチ予測のサンプルと予測されたラベルを含む `predictions` テーブルがあります。バッチ予測でエラーが発生した場合、`errors` テーブルも表示されます。`errors` テーブルには予測に失敗した行が含まれます。

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.